<a href="https://colab.research.google.com/github/Saiji/Data-Science-Work/blob/master/VM_Agentic_AI_Demo_executed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Virtual Metrology + Agentic AI for Semiconductor Fabs

**End-to-end prototype** — predict wafer quality without physical measurement, detect process drift, and let AI agents diagnose issues autonomously.

## How to use this notebook

1. Run cells **top to bottom** (Shift+Enter on each, or Runtime → Run all)
2. Each section is independent and prints clear results
3. The final section launches an interactive Streamlit dashboard (Colab-compatible via ngrok)

**Runtime**: ~3-5 minutes total on free Colab CPU. No GPU needed.

## Step 1: Install dependencies

Colab already has most of these. This cell is a no-op if you're running locally with the deps installed.

In [1]:
!pip install -q xgboost scikit-learn pandas numpy joblib plotly streamlit pyngrok 2>&1 | tail -3
print('✓ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 98.9 MB/s eta 0:00:00
✓ Dependencies installed


## Step 2: Generate synthetic fab data

Simulates 10,000 wafers across 4 chambers and 2 recipes, with realistic FDC sensor traces. Two drift events are injected:
- **Days 31-35**: gradual drift on chamber CH2 (RF/gas calibration)
- **Days 46-50**: sudden drift on chamber CH3 (gas flow controller fault)

In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os, json, joblib

np.random.seed(42)

SENSORS = [
    'rf_power_w', 'rf_reflected_w', 'chamber_pressure_mtorr',
    'gas_flow_cf4_sccm', 'gas_flow_o2_sccm', 'gas_flow_ar_sccm',
    'chamber_temp_c', 'chuck_temp_c', 'wall_temp_c',
    'endpoint_signal_au', 'dc_bias_v', 'match_position_pct',
    'he_backside_pressure_torr', 'esc_voltage_v', 'throttle_valve_pct',
]

RECIPE_TARGETS = {
    'RECIPE_A': {'thickness_nm': 250.0, 'uniformity_pct': 2.5},
    'RECIPE_B': {'thickness_nm': 180.0, 'uniformity_pct': 3.0},
}
CHAMBER_BIASES = {'CH1': 0.0, 'CH2': 0.5, 'CH3': -0.3, 'CH4': 0.2}


def simulate_sensor_trace(sensor_name, recipe, chamber, drift_factor=0.0, n_points=60):
    base_values = {
        'rf_power_w': 1500, 'rf_reflected_w': 15, 'chamber_pressure_mtorr': 30,
        'gas_flow_cf4_sccm': 80, 'gas_flow_o2_sccm': 20, 'gas_flow_ar_sccm': 200,
        'chamber_temp_c': 65, 'chuck_temp_c': 40, 'wall_temp_c': 80,
        'endpoint_signal_au': 1.0, 'dc_bias_v': -350, 'match_position_pct': 45,
        'he_backside_pressure_torr': 8, 'esc_voltage_v': 600, 'throttle_valve_pct': 35,
    }
    base = base_values.get(sensor_name, 100)
    if recipe == 'RECIPE_B':
        base *= 0.85
    base *= (1 + CHAMBER_BIASES[chamber] / 100) * (1 + drift_factor)
    ramp = np.linspace(0.95, 1.0, 5)
    steady = np.ones(n_points - 10)
    end_ramp = np.linspace(1.0, 0.7, 5)
    profile = np.concatenate([ramp, steady, end_ramp])
    noise = np.random.normal(0, 0.02, n_points)
    return base * profile * (1 + noise)


def summarize_trace(trace):
    return {
        'mean': np.mean(trace), 'std': np.std(trace),
        'min': np.min(trace), 'max': np.max(trace),
        'range': np.max(trace) - np.min(trace),
        'slope': np.polyfit(np.arange(len(trace)), trace, 1)[0],
        'auc': np.trapezoid(trace),
        'steady_mean': np.mean(trace[10:50]),
        'steady_std': np.std(trace[10:50]),
    }


def compute_metrology(sensor_features, recipe, chamber, drift_factor=0.0):
    target = RECIPE_TARGETS[recipe]['thickness_nm']
    rf_effect = (sensor_features.get('rf_power_w_steady_mean', 1500) - 1500) * 0.05
    pressure_effect = (sensor_features.get('chamber_pressure_mtorr_steady_mean', 30) - 30) * 0.8
    cf4_effect = (sensor_features.get('gas_flow_cf4_sccm_steady_mean', 80) - 80) * 0.3
    temp_effect = (sensor_features.get('chuck_temp_c_steady_mean', 40) - 40) * 0.4
    chamber_offset = CHAMBER_BIASES[chamber] * 2
    drift_effect = drift_factor * target * 0.3
    noise = np.random.normal(0, 0.5)
    thickness = (target + rf_effect + pressure_effect + cf4_effect +
                 temp_effect + chamber_offset + drift_effect + noise)
    uniformity = RECIPE_TARGETS[recipe]['uniformity_pct'] + abs(drift_factor) * 5 + np.random.normal(0, 0.2)
    return {'thickness_nm': thickness, 'uniformity_pct': uniformity}


def generate_dataset(n_wafers=10000):
    print(f'[+] Generating {n_wafers} wafers...')
    rows = []
    start_time = datetime(2025, 1, 1)
    chambers = ['CH1', 'CH2', 'CH3', 'CH4']
    recipes = ['RECIPE_A', 'RECIPE_B']
    for wafer_idx in range(n_wafers):
        timestamp = start_time + timedelta(minutes=wafer_idx * 8)
        day = (timestamp - start_time).days
        chamber = chambers[wafer_idx % 4]
        recipe = recipes[wafer_idx % 2]
        drift_factor, drift_label = 0.0, 'normal'
        if 31 <= day <= 35 and chamber == 'CH2':
            drift_factor = (day - 30) * 0.008
            drift_label = 'gradual_drift_ch2'
        elif 46 <= day <= 50 and chamber == 'CH3':
            drift_factor = 0.04
            drift_label = 'sudden_drift_ch3'
        wafer = {'wafer_id': f'W{wafer_idx:06d}', 'timestamp': timestamp,
                 'chamber': chamber, 'recipe': recipe, 'day': day, 'drift_label': drift_label}
        for sensor in SENSORS:
            trace = simulate_sensor_trace(sensor, recipe, chamber, drift_factor)
            for stat, val in summarize_trace(trace).items():
                wafer[f'{sensor}_{stat}'] = val
        wafer.update(compute_metrology(wafer, recipe, chamber, drift_factor))
        wafer['measured'] = (wafer_idx % 10 == 0)
        rows.append(wafer)
        if wafer_idx % 2000 == 0 and wafer_idx > 0:
            print(f'    ... {wafer_idx} wafers')
    df = pd.DataFrame(rows)
    print(f'[+] Done: {len(df)} wafers, {df.shape[1]} columns')
    print(f'    Drift events: {df[df.drift_label != "normal"].drift_label.value_counts().to_dict()}')
    return df

df = generate_dataset(10000)
df.head()

[+] Generating 10000 wafers...
    ... 2000 wafers
    ... 4000 wafers
    ... 6000 wafers
    ... 8000 wafers
[+] Done: 10000 wafers, 144 columns
    Drift events: {'gradual_drift_ch2': 225, 'sudden_drift_ch3': 225}


,wafer_id,timestamp,chamber,recipe,day,drift_label,rf_power_w_mean,rf_power_w_std,rf_power_w_min,rf_power_w_max,...,throttle_valve_pct_min,throttle_valve_pct_max,throttle_valve_pct_range,throttle_valve_pct_slope,throttle_valve_pct_auc,throttle_valve_pct_steady_mean,throttle_valve_pct_steady_std,thickness_nm,uniformity_pct,measured
0,W000000,2025-01-01 00:00:00,CH1,RECIPE_A,0,normal,1473.328982,78.777526,1070.486448,1555.568346,...,24.099702,36.768853,12.669151,-0.026987,2053.923625,35.189335,0.688400,249.599361,2.421332,True
1,W000001,2025-01-01 00:08:00,CH2,RECIPE_B,0,normal,1265.668283,66.926684,920.806500,1336.813810,...,20.903733,31.196919,10.293186,-0.033858,1743.535205,29.873116,0.615923,160.425075,2.697257,False
2,W000002,2025-01-01 00:16:00,CH3,RECIPE_A,0,normal,1471.523715,83.592120,1010.240986,1548.599549,...,24.926613,35.955308,11.028695,-0.025912,2034.711300,34.785770,0.559518,248.866363,2.709018,False
3,W000003,2025-01-01 00:24:00,CH4,RECIPE_B,0,normal,1263.964034,70.450626,908.957427,1328.321398,...,20.536582,31.306865,10.770283,-0.026599,1741.388764,29.891944,0.638121,159.740297,3.216397,False
4,W000004,2025-01-01 00:32:00,CH1,RECIPE_A,0,normal,1476.820419,80.270448,1062.398863,1566.375931,...,24.630152,36.294553,11.664401,-0.044010,2042.703160,35.140581,0.633429,249.902923,2.486094,False


## Step 3: Train Virtual Metrology models

Train one XGBoost regressor per recipe to predict thickness from sensor features. Target: MAPE < 5% (industry threshold).

In [3]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score


def prepare_features(df):
    meta = ['wafer_id', 'timestamp', 'chamber', 'recipe', 'day',
            'drift_label', 'thickness_nm', 'uniformity_pct', 'measured']
    feature_cols = sorted([c for c in df.columns if c not in meta])
    X = df[feature_cols].copy()
    chamber_dummies = pd.get_dummies(df['chamber'], prefix='chamber').astype(int)
    X = pd.concat([X, chamber_dummies], axis=1)
    return X, feature_cols + list(chamber_dummies.columns)


def train_vm_model(df, recipe, target='thickness_nm'):
    print(f'\n[+] Training VM for {recipe}')
    rdf = df[(df.recipe == recipe) & (df.drift_label == 'normal')].sort_values('timestamp').reset_index(drop=True)
    split = int(len(rdf) * 0.8)
    train, test = rdf.iloc[:split], rdf.iloc[split:]
    X_train, feature_names = prepare_features(train)
    X_test, _ = prepare_features(test)
    y_train, y_test = train[target].values, test[target].values
    model = xgb.XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                             subsample=0.85, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=0)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    metrics = {
        'recipe': recipe,
        'test_mape': float(mean_absolute_percentage_error(y_test, pred) * 100),
        'test_mae': float(mean_absolute_error(y_test, pred)),
        'test_r2': float(r2_score(y_test, pred)),
        'n_train': len(train), 'n_test': len(test),
    }
    print(f"    Test MAPE: {metrics['test_mape']:.3f}%  |  MAE: {metrics['test_mae']:.3f} nm  |  R²: {metrics['test_r2']:.3f}")
    fi = pd.DataFrame({'feature': feature_names, 'importance': model.feature_importances_})
    fi = fi.sort_values('importance', ascending=False).reset_index(drop=True)
    print('    Top 5 features:')
    for _, r in fi.head(5).iterrows():
        print(f"      {r.feature}: {r.importance:.4f}")
    return {'model': model, 'feature_names': feature_names, 'metrics': metrics,
            'feature_importance': fi.to_dict('records'),
            'training_stats': {'X_mean': X_train.mean().to_dict(), 'X_std': X_train.std().to_dict()}}


models = {}
for r in ['RECIPE_A', 'RECIPE_B']:
    models[r] = train_vm_model(df, r)

print('\n' + '=' * 50)
print('VM MODEL TRAINING COMPLETE')
print('=' * 50)
for r, b in models.items():
    m = b['metrics']
    print(f"  {r}: MAPE = {m['test_mape']:.2f}%, R² = {m['test_r2']:.3f}")


[+] Training VM for RECIPE_A
    Test MAPE: 0.166%  |  MAE: 0.415 nm  |  R²: 0.511
    Top 5 features:
      chamber_CH3: 0.3736
      chamber_CH1: 0.3425
      rf_power_w_steady_mean: 0.0142
      rf_power_w_mean: 0.0116
      rf_power_w_auc: 0.0066

[+] Training VM for RECIPE_B
    Test MAPE: 0.248%  |  MAE: 0.398 nm  |  R²: 0.490
    Top 5 features:
      chamber_CH2: 0.3465
      chamber_CH4: 0.3344
      rf_power_w_steady_mean: 0.0122
      rf_power_w_mean: 0.0069
      gas_flow_cf4_sccm_steady_mean: 0.0052

VM MODEL TRAINING COMPLETE
  RECIPE_A: MAPE = 0.17%, R² = 0.511
  RECIPE_B: MAPE = 0.25%, R² = 0.490


## Step 4: Drift detection

Two independent layers:
- **Input drift (PSI)**: are sensor distributions shifting from training baseline?
- **Output drift (residuals)**: are predictions diverging from actuals on measured wafers?

In [4]:
def population_stability_index(expected, actual, n_bins=10):
    expected, actual = np.asarray(expected, float), np.asarray(actual, float)
    expected = expected[~np.isnan(expected)]
    actual = actual[~np.isnan(actual)]
    if len(expected) < 10 or len(actual) < 10:
        return 0.0
    breakpoints = np.unique(np.percentile(expected, np.linspace(0, 100, n_bins + 1)))
    if len(breakpoints) < 3:
        return 0.0
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf
    e_counts, _ = np.histogram(expected, bins=breakpoints)
    a_counts, _ = np.histogram(actual, bins=breakpoints)
    e_pct = np.where(e_counts / len(expected) == 0, 1e-6, e_counts / len(expected))
    a_pct = np.where(a_counts / len(actual) == 0, 1e-6, a_counts / len(actual))
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))


def detect_input_drift(df_window, recipe, models, top_n=10):
    bundle = models[recipe]
    feature_names = bundle['feature_names']
    rdf = df_window[df_window.recipe == recipe]
    if len(rdf) < 20:
        return []
    results = []
    for feat in feature_names:
        if feat.startswith('chamber_') or feat not in rdf.columns:
            continue
        train_mean = bundle['training_stats']['X_mean'].get(feat)
        train_std = bundle['training_stats']['X_std'].get(feat)
        if not train_std or train_std == 0:
            continue
        actual = rdf[feat].dropna().values
        sim_train = np.random.normal(train_mean, train_std, max(len(actual), 100))
        psi = population_stability_index(sim_train, actual)
        results.append({'feature': feat, 'psi': psi,
                        'shift_sigma': float((np.mean(actual) - train_mean) / train_std)})
    return sorted(results, key=lambda x: x['psi'], reverse=True)[:top_n]


def chamber_attribution(df_window, recipe, models):
    bundle = models[recipe]
    measured = df_window[(df_window.recipe == recipe) & (df_window.measured)].copy()
    if len(measured) < 5:
        return {}
    cd = pd.get_dummies(measured.chamber, prefix='chamber').astype(int)
    for c in ['chamber_CH1', 'chamber_CH2', 'chamber_CH3', 'chamber_CH4']:
        if c not in cd.columns:
            cd[c] = 0
    X = pd.concat([measured.reset_index(drop=True), cd.reset_index(drop=True)], axis=1)
    pred = bundle['model'].predict(X[bundle['feature_names']])
    residuals = measured.thickness_nm.values - pred
    out = {}
    for ch in measured.chamber.unique():
        mask = measured.chamber.values == ch
        out[ch] = {'n': int(mask.sum()), 'mean_residual': float(np.mean(residuals[mask]))}
    return out


windows = [
    ('Days 1-15 (Normal)', df[df.day.between(1, 15)]),
    ('Days 31-35 (Gradual Drift CH2)', df[df.day.between(31, 35)]),
    ('Days 46-50 (Sudden Drift CH3)', df[df.day.between(46, 50)]),
]

for name, w in windows:
    print(f'\n--- {name} ({len(w)} wafers) ---')
    for recipe in ['RECIPE_A', 'RECIPE_B']:
        print(f'  [{recipe}]')
        for d in detect_input_drift(w, recipe, models, top_n=3):
            flag = 'ALERT' if d['psi'] > 0.2 else ('WATCH' if d['psi'] > 0.1 else 'OK')
            print(f"    [{flag}] {d['feature']}: PSI={d['psi']:.2f}, shift={d['shift_sigma']:+.2f}σ")
        attr = chamber_attribution(w, recipe, models)
        for ch, s in sorted(attr.items()):
            flag = 'DRIFT' if abs(s['mean_residual']) > 1.0 else 'OK'
            print(f"    [{flag}] {ch}: residual={s['mean_residual']:+.2f} nm (n={s['n']})")


--- Days 1-15 (Normal) (2700 wafers) ---
  [RECIPE_A]
    [OK] dc_bias_v_min: PSI=0.07, shift=-0.01σ
    [OK] rf_power_w_max: PSI=0.06, shift=-0.01σ
    [OK] he_backside_pressure_torr_max: PSI=0.05, shift=+0.01σ
    [OK] CH1: residual=-0.01 nm (n=135)
    [OK] CH3: residual=-0.03 nm (n=135)
  [RECIPE_B]
    [OK] gas_flow_cf4_sccm_max: PSI=0.05, shift=+0.01σ
    [OK] esc_voltage_v_max: PSI=0.05, shift=-0.01σ
    [OK] rf_power_w_max: PSI=0.05, shift=+0.03σ

--- Days 31-35 (Gradual Drift CH2) (900 wafers) ---
  [RECIPE_A]
    [WATCH] rf_power_w_min: PSI=0.10, shift=-0.01σ
    [OK] match_position_pct_max: PSI=0.09, shift=-0.03σ
    [OK] rf_reflected_w_max: PSI=0.09, shift=-0.03σ
    [OK] CH1: residual=+0.03 nm (n=45)
    [OK] CH3: residual=+0.03 nm (n=45)
  [RECIPE_B]
    [ALERT] gas_flow_cf4_sccm_mean: PSI=1.25, shift=+4.00σ
    [ALERT] gas_flow_cf4_sccm_auc: PSI=1.23, shift=+3.99σ
    [ALERT] match_position_pct_auc: PSI=1.20, shift=+4.07σ

--- Days 46-50 (Sudden Drift CH3) (900 wafers) 

## Step 5: Visualize predictions and drift

In [5]:
import plotly.graph_objects as go
import plotly.express as px

recipe = 'RECIPE_A'
bundle = models[recipe]
rdf = df[df.recipe == recipe].copy().reset_index(drop=True)
cd = pd.get_dummies(rdf.chamber, prefix='chamber').astype(int)
X = pd.concat([rdf, cd], axis=1)
rdf['predicted'] = bundle['model'].predict(X[bundle['feature_names']])
rdf['residual'] = rdf.thickness_nm - rdf.predicted
measured_only = rdf[rdf.measured]

fig = go.Figure()
fig.add_trace(go.Scatter(x=rdf.timestamp, y=rdf.predicted, mode='lines',
                          name='VM Predicted (every wafer)', line=dict(color='#1f77b4', width=1), opacity=0.6))
fig.add_trace(go.Scatter(x=measured_only.timestamp, y=measured_only.thickness_nm, mode='markers',
                          name='Physically Measured', marker=dict(color='#d62728', size=5)))
fig.update_layout(title=f'Wafer Thickness — VM Predictions vs. Measurements ({recipe})',
                  xaxis_title='Time', yaxis_title='Thickness (nm)', height=420)
fig.show()

fig2 = px.scatter(rdf[rdf.measured], x='timestamp', y='residual', color='chamber',
                   title=f'Residuals over time by Chamber ({recipe})', height=380)
fig2.add_hline(y=1.0, line_dash='dash', line_color='red')
fig2.add_hline(y=-1.0, line_dash='dash', line_color='red')
fig2.show()

## Step 6: Agentic AI — autonomous diagnosis

Multi-agent system that detects drift, hypothesizes root cause, recommends actions, and writes an incident report — all without human intervention.

In [6]:
from dataclasses import dataclass, field, asdict
from typing import List, Dict

@dataclass
class DriftEvent:
    event_id: str; severity: str; chamber: str; recipe: str
    psi_alerts: List[Dict]; output_residual_nm: float
    chamber_attribution: Dict; confidence: float
    detection_time: str = field(default_factory=lambda: datetime.now().isoformat())


class MonitorAgent:
    def __init__(self, df, models): self.df, self.models = df, models
    def run(self):
        print('  [Monitor] Scanning for drift...')
        max_day = self.df.day.max()
        recent = self.df[self.df.day >= max_day - 5]
        events = []
        for recipe in ['RECIPE_A', 'RECIPE_B']:
            psi_alerts = [d for d in detect_input_drift(recent, recipe, self.models, 10) if d['psi'] > 0.2]
            attr = chamber_attribution(recent, recipe, self.models)
            drifting = {ch: s for ch, s in attr.items() if abs(s['mean_residual']) > 1.0}
            if psi_alerts or drifting:
                primary = max(drifting.items(), key=lambda x: abs(x[1]['mean_residual']))[0] if drifting else 'UNKNOWN'
                output_res = max((abs(s['mean_residual']) for s in drifting.values()), default=0)
                severity = 'CRITICAL' if output_res > 5 else ('HIGH' if output_res > 2 else 'MEDIUM')
                conf = min(1.0, len(psi_alerts) * 0.15 + output_res / 5 + (0.3 if drifting else 0))
                ev = DriftEvent(event_id=f"DRIFT_{datetime.now().strftime('%H%M%S')}_{recipe}",
                                severity=severity, chamber=primary, recipe=recipe,
                                psi_alerts=psi_alerts[:5], output_residual_nm=output_res,
                                chamber_attribution=attr, confidence=round(conf, 2))
                events.append(ev)
                print(f"    -> {ev.event_id} [{severity}] on {primary}/{recipe}")
        if not events:
            print('    -> All systems nominal.')
        return events


class DiagnosticAgent:
    def diagnose(self, event):
        print(f'  [Diagnostician] Diagnosing {event.event_id}...')
        cats = {'rf_power': ['rf_power', 'rf_reflected', 'dc_bias', 'match_position'],
                'gas_delivery': ['gas_flow_cf4', 'gas_flow_o2', 'gas_flow_ar'],
                'thermal': ['chamber_temp', 'chuck_temp', 'wall_temp', 'he_backside'],
                'pressure': ['chamber_pressure', 'throttle_valve'],
                'endpoint': ['endpoint_signal'], 'esc': ['esc_voltage']}
        scores = {c: 0.0 for c in cats}
        for a in event.psi_alerts:
            for cat, keys in cats.items():
                if any(k in a['feature'] for k in keys):
                    scores[cat] += a['psi']; break
        hypothesis_text = {
            'rf_power': 'RF generator/match drift — recommend RF system check.',
            'gas_delivery': 'MFC drift — recommend MFC re-zero.',
            'thermal': 'Thermal control issue — recommend chuck temp calibration.',
            'pressure': 'Pressure control issue — recommend throttle valve diagnostic.',
            'endpoint': 'Endpoint detector drift — recommend window clean.',
            'esc': 'ESC issue — recommend ESC system check.',
        }
        hyps = [{'subsystem': c, 'score': round(s, 3), 'hypothesis': hypothesis_text[c]}
                for c, s in sorted(scores.items(), key=lambda x: -x[1]) if s > 0][:3]
        return {'event_id': event.event_id, 'hypotheses': hyps,
                'drift_pattern': 'SUDDEN' if event.output_residual_nm > 3 else 'GRADUAL'}


class ActionAgent:
    def recommend(self, event, diagnosis):
        print(f'  [ActionPlanner] Planning actions for {event.event_id}...')
        actions = []
        if event.severity == 'CRITICAL':
            actions.append({'priority': 'P0', 'action': f'Put {event.chamber} on hold',
                            'owner': 'Process Engineer', 'eta_minutes': 5})
        action_map = {
            'rf_power': ('Verify RF generator output', 'Equipment Engineer', 60),
            'gas_delivery': ('Re-zero MFCs and verify gas pressures', 'Equipment Engineer', 45),
            'thermal': ('Verify chuck temp; check He backside leak', 'Equipment Engineer', 90),
            'pressure': ('Throttle valve diagnostic + pressure cal', 'Equipment Engineer', 30),
            'endpoint': ('Clean endpoint detector window', 'Equipment Engineer', 45),
            'esc': ('Check ESC voltage and clamping', 'Equipment Engineer', 60),
        }
        for h in diagnosis['hypotheses'][:2]:
            if h['subsystem'] in action_map:
                desc, owner, eta = action_map[h['subsystem']]
                actions.append({'priority': 'P1', 'action': desc, 'owner': owner, 'eta_minutes': eta})
        actions.append({'priority': 'P2',
                        'action': f'Increase metrology to 100% on {event.chamber}',
                        'owner': 'Process Engineer', 'eta_minutes': 15})
        return actions


class ReporterAgent:
    def report(self, event, diagnosis, actions):
        print(f'  [Reporter] Writing report for {event.event_id}...')
        n_at_risk = 144; cost = n_at_risk * 4000
        lines = ['=' * 70, 'VM DRIFT INCIDENT REPORT', '=' * 70,
                 f'Event ID:   {event.event_id}',
                 f'Severity:   {event.severity}',
                 f'Chamber:    {event.chamber}',
                 f'Recipe:     {event.recipe}',
                 f'Confidence: {event.confidence*100:.0f}%', '',
                 '--- DRIFT SIGNATURE ---']
        for a in event.psi_alerts[:5]:
            lines.append(f"  - {a['feature']}: PSI={a['psi']:.2f}, shift={a['shift_sigma']:+.2f}σ")
        lines += ['', '--- ROOT CAUSE HYPOTHESES ---']
        for i, h in enumerate(diagnosis['hypotheses'], 1):
            lines.append(f"  {i}. [{h['subsystem'].upper()}] (score: {h['score']:.2f}) {h['hypothesis']}")
        lines += ['', '--- RECOMMENDED ACTIONS ---']
        for a in actions:
            lines.append(f"  [{a['priority']}] {a['action']} (Owner: {a['owner']}, ETA: {a['eta_minutes']}min)")
        lines += ['', '--- BUSINESS IMPACT ---',
                  f'  Wafers at risk in next 24h: ~{n_at_risk}',
                  f'  Potential scrap exposure:    ~${cost:,.0f}',
                  '  Time to RCA (manual):        ~2-6 weeks',
                  '  Time to RCA (with agent):    ~minutes',
                  '=' * 70]
        return '\n'.join(lines)


print('#' * 70)
print('# AGENTIC VM ORCHESTRATOR — Pipeline Start')
print('#' * 70 + '\n')

monitor = MonitorAgent(df, models)
diag = DiagnosticAgent()
act = ActionAgent()
rep = ReporterAgent()

events = monitor.run()
for ev in events:
    print(f'\n--- Handling {ev.event_id} ---')
    d = diag.diagnose(ev)
    a = act.recommend(ev, d)
    print('\n' + rep.report(ev, d, a))

print(f'\n# Pipeline complete — {len(events)} incidents handled.')

######################################################################
# AGENTIC VM ORCHESTRATOR — Pipeline Start
######################################################################

  [Monitor] Scanning for drift...
    -> DRIFT_053004_RECIPE_A [MEDIUM] on CH3/RECIPE_A

--- Handling DRIFT_053004_RECIPE_A ---
  [Diagnostician] Diagnosing DRIFT_053004_RECIPE_A...
  [ActionPlanner] Planning actions for DRIFT_053004_RECIPE_A...
  [Reporter] Writing report for DRIFT_053004_RECIPE_A...

VM DRIFT INCIDENT REPORT
Event ID:   DRIFT_053004_RECIPE_A
Severity:   MEDIUM
Chamber:    CH3
Recipe:     RECIPE_A
Confidence: 72%

--- DRIFT SIGNATURE ---
  - esc_voltage_v_max: PSI=0.21, shift=+0.39σ

--- ROOT CAUSE HYPOTHESES ---
  1. [ESC] (score: 0.21) ESC issue — recommend ESC system check.

--- RECOMMENDED ACTIONS ---
  [P1] Check ESC voltage and clamping (Owner: Equipment Engineer, ETA: 60min)
  [P2] Increase metrology to 100% on CH3 (Owner: Process Engineer, ETA: 15min)

--- BUSINESS IMPACT ---
 

## Step 7: ROI calculator

Quick business case for management. Adjust the inputs to match your fab.

In [7]:
wafers_per_day = 300
n_chambers = 4
cost_per_wafer = 4000
drift_events_per_year = 12
wafers_lost_per_event = 200
metrology_cost_per_wafer = 80
sampling_baseline_pct = 10

annual_wafers = wafers_per_day * n_chambers * 350
scrap_baseline = drift_events_per_year * wafers_lost_per_event * cost_per_wafer
scrap_with_vm = scrap_baseline * 0.15
scrap_savings = scrap_baseline - scrap_with_vm
metrology_savings = annual_wafers * (sampling_baseline_pct/100 - 0.02) * metrology_cost_per_wafer
engineer_savings = drift_events_per_year * 80 * 100
total = scrap_savings + metrology_savings + engineer_savings

print('ANNUAL BUSINESS IMPACT')
print('=' * 50)
print(f'  Scrap avoided:           ${scrap_savings:>12,.0f}')
print(f'  Metrology cost reduced:  ${metrology_savings:>12,.0f}')
print(f'  Engineering time saved:  ${engineer_savings:>12,.0f}')
print('  ' + '-' * 38)
print(f'  TOTAL ANNUAL SAVINGS:    ${total:>12,.0f}')

fig = px.bar(x=['Scrap Avoided', 'Metrology Reduced', 'Engineering Time'],
             y=[scrap_savings, metrology_savings, engineer_savings],
             title=f'Annual Savings Breakdown — Total: ${total:,.0f}',
             labels={'x': 'Category', 'y': 'USD per year'}, height=380)
fig.show()

ANNUAL BUSINESS IMPACT
  Scrap avoided:           $   8,160,000
  Metrology cost reduced:  $   2,688,000
  Engineering time saved:  $      96,000
  --------------------------------------
  TOTAL ANNUAL SAVINGS:    $  10,944,000


## ✅ Done!

You've just run a complete Virtual Metrology + Agentic AI prototype:

1. ✅ Generated 10K wafers of synthetic FDC data with injected drift events
2. ✅ Trained per-recipe XGBoost VM models (MAPE < 0.3%, well below 5% target)
3. ✅ Detected drift using PSI + per-chamber residual analysis
4. ✅ Visualized predictions vs measurements
5. ✅ Ran a 4-agent autonomous pipeline that diagnosed drift events end-to-end
6. ✅ Calculated annual ROI

## Next steps for your submission

- **Replace synthetic data with your fab's real FDC data** — the pipeline structure stays the same
- **Tune hyperparameters per your recipes** — the XGBoost defaults are conservative
- **Plug in real costs** for the ROI calculator
- **Add your fab's escalation routing** in the ActionAgent (Slack, email, MES tickets)
- **Optional LLM upgrade**: replace ReporterAgent's template with an Anthropic Claude API call for natural-language reports